In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
import uuid
import ast
import pandas as pd

In [15]:
df = pd.read_csv("drive/MyDrive/GenAI/proj/reduced_newsamp.csv")

In [16]:
df.columns

Index(['title', 'orchestra', 'singer', 'year', 'decade', 'style',
       'duration_seconds', 'combo_key', 'album', 'filename', 'bpm',
       'bpm_label', 'danceability', 'danceability_label', 'key',
       'chords_changes_rate', 'chords_changes_rate_label', 'energy_composite',
       'energy_label', 'jamendo_tags', 'mirex_tags', 'mood_tags'],
      dtype='object')

In [19]:
# Generate unique id
df["id"] = df.apply(
    lambda r: str(uuid.uuid5(uuid.NAMESPACE_DNS, f"{r['album']}|{r['filename']}")),
    axis=1
)

# Generate file_path
df["file_path"] = "data/raw/" + df["album"].astype(str) + "/" + df["filename"].astype(str)

# Energy
df["energy"] = df["energy_composite"]

# Combine tag columns into one list column
def ensure_list(x):
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        x = x.strip()
        if not x:
            return []
        try:
            parsed = ast.literal_eval(x)
            if isinstance(parsed, list):
                return parsed
        except Exception:
            pass
        return [x]
    return [x]

df["tags"] = df.apply(
    lambda r: (
        ensure_list(r["jamendo_tags"])
        + ensure_list(r["mirex_tags"])
        + ensure_list(r["mood_tags"])
    ),
    axis=1
)

catalog_cols = [
    'id', 'title', 'orchestra', 'singer', 'year', 'decade', 'style',
    'duration_seconds', 'combo_key', 'album', 'filename', 'bpm',
    'bpm_label', 'danceability', 'danceability_label', 'key',
    'chords_changes_rate', 'chords_changes_rate_label', 'energy', 'energy_label',
    'tags', 'file_path'
]

catalog = df[catalog_cols].copy()

catalog.to_csv("drive/MyDrive/GenAI/proj/rag_catalog.csv", index=False)

In [20]:
catalog

,id,title,orchestra,singer,year,decade,style,duration_seconds,combo_key,album,...,bpm_label,danceability,danceability_label,key,chords_changes_rate,chords_changes_rate_label,energy,energy_label,tags,file_path
0,1fa67d03-8d23-507d-b16e-f181e5fe7b86,Asi Me Gusta A Mi,Angel D'Agostino,Angel Vargas,1942.0,1940s,milonga,168.95,angel d'agostino | angel vargas | milonga,vol-21 la fiesta de buenos aires,...,slow,1.016401,moderate,D,0.083173,high,0.427760,low,"[humorous, silly, campy, quirky, whimsical, wi...",data/raw/vol-21 la fiesta de buenos aires/27 a...
1,5ce62a75-7084-5a90-84d7-85a7a5dcb607,Compadreando,Angel D'Agostino,Angel Vargas,1941.0,1940s,milonga,152.95,angel d'agostino | angel vargas | milonga,vol-21 la fiesta de buenos aires,...,slow,1.020687,moderate,D,0.062765,moderate,0.740297,high,"[reggae, drums, popfolk, rollicking, cheerful,...",data/raw/vol-21 la fiesta de buenos aires/28 c...
2,648048b8-8d53-54ad-901b-af1c18f4c3cd,En Lo de Laura,Angel D'Agostino,Angel Vargas,1943.0,1940s,milonga,142.36,angel d'agostino | angel vargas | milonga,vol-06 la fiesta de buenos aires,...,slow,1.063536,high,D,0.081653,high,0.648099,moderate,"[drums, humorous, silly, campy, quirky, whimsi...",data/raw/vol-06 la fiesta de buenos aires/26 e...
3,aa41c125-1439-5c43-9e65-9101248b31f6,Entre Copa y Copa,Angel D'Agostino,Angel Vargas,1942.0,1940s,milonga,150.28,angel d'agostino | angel vargas | milonga,vol-06 la fiesta de buenos aires,...,slow,1.079122,high,G,0.076425,high,0.759466,high,"[rollicking, cheerful, fun, sweet, amiable, go...",data/raw/vol-06 la fiesta de buenos aires/28 e...
4,ed820a5a-e298-53ba-9561-6fa9d96aae23,Porque Me Siento Feliz,Angel D'Agostino,Angel Vargas,1945.0,1940s,milonga,151.82,angel d'agostino | angel vargas | milonga,vol-06 la fiesta de buenos aires,...,slow,1.083903,high,C,0.071080,moderate,0.797133,high,"[rollicking, cheerful, fun, sweet, amiable, go...",data/raw/vol-06 la fiesta de buenos aires/29 p...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
289,521dabc6-73e5-5b87-be8f-023f913824de,Che Papusa Oi,Tipica Victor,Instrumental,1958.0,1950s,tango,153.05,tipica victor | instrumental | tango,vol-09 la fiesta de buenos aires,...,very fast,1.166777,high,B,0.048123,low,0.734900,high,"[experimental, humorous, silly, campy, quirky,...",data/raw/vol-09 la fiesta de buenos aires/22 c...
290,0fd41987-8c25-5117-91ab-00a56be4de73,Pato,Tipica Victor,Instrumental,1926.0,1920s,tango,145.20,tipica victor | instrumental | tango,vol-29 la fiesta de buenos aires,...,fast,1.048075,moderate,Bb,0.040243,low,0.404533,low,"[ambient, experimental, humorous, silly, campy...",data/raw/vol-29 la fiesta de buenos aires/08 p...
291,adcd585c-152c-5486-a191-745a8c8b41ef,De Mi Barrio,Tipica Victor,Instrumental,1999.0,1990s,tango,167.20,tipica victor | instrumental | tango,vol-29 la fiesta de buenos aires,...,moderate,1.151076,high,D,0.049653,low,0.659794,moderate,"[jazz, popfolk, experimental, humorous, silly,...",data/raw/vol-29 la fiesta de buenos aires/07 d...
292,298ceee5-0751-5025-9e8b-17a182879974,El Porteñito,Tipica Victor,Instrumental,1949.0,1940s,tango,149.89,tipica victor | instrumental | tango,vol-09 la fiesta de buenos aires,...,very fast,1.181283,high,Eb,0.054697,low,0.572959,moderate,"[jazz, popfolk, experimental, humorous, silly,...",data/raw/vol-09 la fiesta de buenos aires/21 e...
